# Fibonacci Numbers, Recursion Trees, and Memoization

**Session:** 2026-11-02 · Recursion and Two Pointer  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking

The Fibonacci sequence is defined by the recurrence:
$$F(0) = 0, \quad F(1) = 1$$
$$F(n) = F(n-1) + F(n-2) \quad \text{for } n \ge 2$$

Unlike linear recursion where each call generates a single sub-call, Fibonacci generates a **binary recursion tree** where each node spawns two child invocations.


## 1. Naive Binary Recursion: The Exponential Explosion

When computing $F(4)$, the call tree branches as:
- $F(4) \to F(3) + F(2)$
  - $F(3) \to F(2) + F(1)$
    - $F(2) \to F(1) + F(0)$
  - $F(2) \to F(1) + F(0)$

Notice that $F(2)$ is evaluated twice independently, $F(1)$ three times, and $F(0)$ two times. The total number of nodes in the recursion tree grows asymptotically as:
$$T(n) = \Theta(\phi^n) \approx O(1.618^n) \le O(2^n)$$
where $\phi = \frac{1 + \sqrt{5}}{2}$ is the Golden Ratio.


In [ ]:
def fibonacci_naive(n: int) -> int:
    """Compute n-th Fibonacci number naively via binary recursion.

    Time Complexity: O(2^n)
    Auxiliary Space: O(n) call stack depth
    """
    if n < 0:
        raise ValueError("Fibonacci is undefined for negative integers.")
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci_naive(n - 1) + fibonacci_naive(n - 2)


def count_fibonacci_calls(n: int) -> tuple[int, int]:
    """Track total call count during naive evaluation."""
    call_count = 0

    def helper(k: int) -> int:
        nonlocal call_count
        call_count += 1
        if k == 0:
            return 0
        if k == 1:
            return 1
        return helper(k - 1) + helper(k - 2)

    val = helper(n)
    return val, call_count


for n in range(6):
    val, calls = count_fibonacci_calls(n)
    print(f"F({n}) = {val:<3} | Total recursive calls: {calls}")

## 2. Dynamic Programming: Memoization vs Tabulation

We can eliminate redundant computation through two paradigms:
1. **Top-Down Memoization:** Cache subproblem solutions in a hash table or array upon first calculation.
2. **Bottom-Up Tabulation / Iteration:** Compute solutions sequentially from $F(0)$ up to $F(n)$ using two scalar accumulators, achieving $O(n)$ time and $O(1)$ auxiliary space.


In [ ]:
from functools import cache


def fibonacci_memo(n: int, memo: dict[int, int] | None = None) -> int:
    """Top-down memoization using an explicit dictionary.

    Time Complexity: O(n)
    Auxiliary Space: O(n) (memo table + call stack)
    """
    if memo is None:
        memo = {0: 0, 1: 1}
    if n < 0:
        raise ValueError("Fibonacci is undefined for negative integers.")
    if n not in memo:
        memo[n] = fibonacci_memo(n - 1, memo) + fibonacci_memo(n - 2, memo)
    return memo[n]


@cache
def fibonacci_lru(n: int) -> int:
    """Top-down memoization using Python's standard lru_cache."""
    if n < 0:
        raise ValueError("Fibonacci is undefined for negative integers.")
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci_lru(n - 1) + fibonacci_lru(n - 2)


def fibonacci_iterative(n: int) -> int:
    """Bottom-up iterative approach with constant auxiliary space.

    Time Complexity: O(n)
    Auxiliary Space: O(1)
    """
    if n < 0:
        raise ValueError("Fibonacci is undefined for negative integers.")
    if n == 0:
        return 0
    prev, curr = 0, 1
    for _ in range(2, n + 1):
        prev, curr = curr, prev + curr
    return curr


assert fibonacci_memo(10) == 55
assert fibonacci_lru(10) == 55
assert fibonacci_iterative(10) == 55
print("F(30) via memoization:", fibonacci_memo(30))
print("F(30) via iteration:", fibonacci_iterative(30))

## 3. Case Study: The Fibonacci Word Sequence (Slide 64)

The **Fibonacci Word sequence** is generated by string concatenation:
$$F(0) = \text{"A"}, \quad F(1) = \text{"B"}$$
$$F(n) = F(n-1) + F(n-2) \quad \text{for } n \ge 2$$

Sequence progression:
- $F(2) = F(1) + F(0) = \text{"BA"} \implies |F(2)| = 2$
- $F(3) = F(2) + F(1) = \text{"BAB"} \implies |F(3)| = 3$
- $F(4) = F(3) + F(2) = \text{"BABBA"} \implies |F(4)| = 5$
- $F(5) = F(4) + F(3) = \text{"BABBABAB"} \implies |F(5)| = 8$

**Problem Statement (Slide 64):**
Given an integer `target`, return the character indexed at `(target - 1)` in string $F(n)$, where $n$ is the smallest integer such that $|F(n)| \ge \text{target}$.

Example: `target = 4` $\implies n = 4$ because $|F(4)| = 5 \ge 4$, and $F(4)[4 - 1] = F(4)[3] = \text{'B'}$.

Rather than constructing astronomical strings in memory, we can navigate the recurrence tree recursively based on prefix lengths.


In [ ]:
import json
from pathlib import Path


def fibonacci_word_length(n: int) -> int:
    """Length of n-th Fibonacci word: L(0)=1, L(1)=1, L(n)=L(n-1)+L(n-2)."""
    if n <= 1:
        return 1
    a, b = 1, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


def generate_fibonacci_word(n: int) -> str:
    """Generate n-th Fibonacci word directly via string concatenation."""
    if n == 0:
        return "A"
    if n == 1:
        return "B"
    prev, curr = "A", "B"
    for _ in range(2, n + 1):
        prev, curr = curr, curr + prev
    return curr


def get_fibonacci_word_char_recursive(target: int) -> tuple[int, str]:
    """Solve Slide 64 efficiently without keeping gigantic strings in memory.

    Time Complexity: O(n)
    Auxiliary Space: O(n) stack frames
    """
    if target <= 0:
        raise ValueError("Target index must be positive.")

    # Step 1: Find smallest n such that length >= target
    n = 0
    while fibonacci_word_length(n) < target:
        n += 1

    # Step 2: Binary recursive navigation down the length hierarchy
    def find_char(k: int, idx: int) -> str:
        if k == 0:
            return "A"
        if k == 1:
            return "B"
        left_len = fibonacci_word_length(k - 1)
        if idx < left_len:
            return find_char(k - 1, idx)
        return find_char(k - 2, idx - left_len)

    char = find_char(n, target - 1)
    return n, char


# Verification against benchmark dataset
data_path = Path("../data/fibonacci_words.json")
if data_path.is_file():
    cases = json.loads(data_path.read_text(encoding="utf-8"))
    for case in cases:
        target = case["target"]
        n_calc, char_calc = get_fibonacci_word_char_recursive(target)
        print(
            f"Target {target}: found n={n_calc}, char='{char_calc}' (Expected: '{case['expected_char']}')"
        )
        assert n_calc == case["closest_n"]
        assert char_calc == case["expected_char"]